# BERT from Scratch — A Beginner-Friendly TensorFlow Notebook

This notebook follows the structure of your Transformer notebook, but builds a **small BERT-style encoder** and trains it with two pretraining objectives:

1. **Masked Language Modeling (MLM):** predict selected masked tokens.
2. **Next Sentence Prediction (NSP):** predict whether sentence B really follows sentence A.

> Educational demo: this is a tiny model trained on a small text file, not Google's full pretrained BERT. It demonstrates the architecture and training idea; it will not reach the quality of a pretrained BERT model.

In [1]:
import re
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

file_path = "/content/Book1.txt"

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:500])

/ 




THE BOY WHO LIVED 

Mr. and Mrs. Dursley, of number four, Privet Drive, 
were proud to say that they were perfectly normal, 
thank you very much. They were the last people you’d 
expect to be involved in anything strange or 
mysterious, because they just didn’t hold with such 
nonsense. 

Mr. Dursley was the director of a firm called 
Grunnings, which made drills. He was a big, beefy 
man with hardly any neck, although he did have a 
very large mustache. Mrs. Dursley was thin and 
blonde 


## 2. Split the text into sentences and build a simple vocabulary

For teaching clarity, this uses a simple word-level tokenizer. Real BERT uses **WordPiece** subword tokenization.

In [2]:
def split_sentences(text):
    text = re.sub(r"\s+", " ", text)
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip().lower() for p in parts if len(p.strip().split()) >= 4]

sentences = split_sentences(raw_text)
print("Number of sentences:", len(sentences))
print(sentences[:5])

# Reserve special tokens
PAD, CLS, SEP, MASK, UNK = "[PAD]", "[CLS]", "[SEP]", "[MASK]", "[UNK]"
special_tokens = [PAD, CLS, SEP, MASK, UNK]

def tokenize(sentence):
    # Simple educational tokenizer; punctuation is separated
    return re.findall(r"\w+|[^\w\s]", sentence.lower())

from collections import Counter
counter = Counter()
for sentence in sentences:
    counter.update(tokenize(sentence))

# Limit vocabulary to keep the demo manageable
MAX_VOCAB = 8000
vocab_words = [w for w, _ in counter.most_common(MAX_VOCAB - len(special_tokens))
               if w not in special_tokens]
vocab = special_tokens + vocab_words
token_to_id = {token: i for i, token in enumerate(vocab)}
id_to_token = {i: token for token, i in token_to_id.items()}

PAD_ID = token_to_id[PAD]
CLS_ID = token_to_id[CLS]
SEP_ID = token_to_id[SEP]
MASK_ID = token_to_id[MASK]
UNK_ID = token_to_id[UNK]
VOCAB_SIZE = len(vocab)

def encode_words(text):
    return [token_to_id.get(tok, UNK_ID) for tok in tokenize(text)]

print("Vocabulary size:", VOCAB_SIZE)
print("Example tokens:", tokenize(sentences[0]))

Number of sentences: 5128
['/ the boy who lived mr.', 'dursley, of number four, privet drive, were proud to say that they were perfectly normal, thank you very much.', 'they were the last people you’d expect to be involved in anything strange or mysterious, because they just didn’t hold with such nonsense.', 'dursley was the director of a firm called grunnings, which made drills.', 'he was a big, beefy man with hardly any neck, although he did have a very large mustache.']
Vocabulary size: 6072
Example tokens: ['/', 'the', 'boy', 'who', 'lived', 'mr', '.']


## 3. Create sentence pairs for NSP

Positive pair: sentence B is the actual next sentence.  
Negative pair: sentence B is a randomly selected sentence.

The input format is `[CLS] sentence A [SEP] sentence B [SEP]`.

In [3]:
MAX_LEN = 64

def make_pair_examples(sentences):
    examples = []
    if len(sentences) < 2:
        raise ValueError("The text file must contain at least two usable sentences.")

    for i in range(len(sentences) - 1):
        sent_a = sentences[i]
        is_next = random.random() < 0.5

        if is_next:
            sent_b = sentences[i + 1]
            nsp_label = 1
        else:
            other_index = random.randrange(len(sentences))
            while other_index in (i, i + 1):
                other_index = random.randrange(len(sentences))
            sent_b = sentences[other_index]
            nsp_label = 0

        a_ids = encode_words(sent_a)
        b_ids = encode_words(sent_b)

        # Keep room for [CLS] and two [SEP] tokens
        while len(a_ids) + len(b_ids) > MAX_LEN - 3:
            if len(a_ids) > len(b_ids):
                a_ids.pop()
            else:
                b_ids.pop()

        input_ids = [CLS_ID] + a_ids + [SEP_ID] + b_ids + [SEP_ID]
        segment_ids = [0] * (len(a_ids) + 2) + [1] * (len(b_ids) + 1)
        return_mask = [1] * len(input_ids)

        # Pad all arrays to MAX_LEN
        pad_count = MAX_LEN - len(input_ids)
        input_ids += [PAD_ID] * pad_count
        segment_ids += [0] * pad_count
        return_mask += [0] * pad_count

        examples.append((input_ids, segment_ids, return_mask, nsp_label))

    return examples

examples = make_pair_examples(sentences)
random.shuffle(examples)
print("Examples:", len(examples))
print("First NSP label (1=IsNext, 0=NotNext):", examples[0][3])

Examples: 5127
First NSP label (1=IsNext, 0=NotNext): 0


## 4. Apply Masked Language Modeling (MLM)

We select about 15% of non-special, non-padding tokens. In the original BERT recipe, selected tokens are replaced using an 80/10/10 rule (MASK/random/original). This simplified demo uses `[MASK]` for selected tokens and predicts their original IDs.

In [4]:
SPECIAL_IDS = {PAD_ID, CLS_ID, SEP_ID, MASK_ID}

def apply_mlm(input_ids, attention_mask, probability=0.15):
    masked_input = list(input_ids)
    mlm_labels = [-100] * len(input_ids)  # -100 means "ignore this position"

    candidates = [i for i, token_id in enumerate(input_ids)
                  if attention_mask[i] == 1 and token_id not in SPECIAL_IDS]

    selected = []
    for pos in candidates:
        if random.random() < probability:
            selected.append(pos)

    # Ensure at least one token is selected when possible
    if not selected and candidates:
        selected = [random.choice(candidates)]

    for pos in selected:
        mlm_labels[pos] = input_ids[pos]
        masked_input[pos] = MASK_ID

    return masked_input, mlm_labels

def prepare_arrays(examples):
    all_input, all_segments, all_attention = [], [], []
    all_mlm_labels, all_nsp = [], []

    for input_ids, segment_ids, attention_mask, nsp_label in examples:
        masked_input, mlm_labels = apply_mlm(input_ids, attention_mask)
        all_input.append(masked_input)
        all_segments.append(segment_ids)
        all_attention.append(attention_mask)
        all_mlm_labels.append(mlm_labels)
        all_nsp.append(nsp_label)

    return (
        np.array(all_input, dtype=np.int32),
        np.array(all_segments, dtype=np.int32),
        np.array(all_attention, dtype=np.int32),
        np.array(all_mlm_labels, dtype=np.int32),
        np.array(all_nsp, dtype=np.float32).reshape(-1, 1),
    )

input_ids, segment_ids, attention_masks, mlm_labels, nsp_labels = prepare_arrays(examples)

split = max(1, int(0.9 * len(input_ids)))
train_data = (input_ids[:split], segment_ids[:split], attention_masks[:split],
              mlm_labels[:split], nsp_labels[:split])
val_data = (input_ids[split:], segment_ids[split:], attention_masks[split:],
            mlm_labels[split:], nsp_labels[split:])

print("Input IDs shape:", input_ids.shape)
print("MLM labels shape:", mlm_labels.shape)
print("NSP labels shape:", nsp_labels.shape)

Input IDs shape: (5127, 64)
MLM labels shape: (5127, 64)
NSP labels shape: (5127, 1)


## 5. Build a BERT-style Transformer encoder

The model adds **token + position + segment embeddings**, then applies bidirectional self-attention. The outputs are used by two heads:
- MLM head: predicts a vocabulary token at each sequence position.
- NSP head: uses the `[CLS]` representation to predict IsNext / NotNext.

In [5]:
class BertEmbeddings(layers.Layer):
    def __init__(self, vocab_size, max_len, hidden_size):
        super().__init__()
        self.token_embedding = layers.Embedding(vocab_size, hidden_size)
        self.position_embedding = layers.Embedding(max_len, hidden_size)
        self.segment_embedding = layers.Embedding(2, hidden_size)
        self.norm = layers.LayerNormalization(epsilon=1e-12)
        self.dropout = layers.Dropout(0.1)

    def call(self, inputs, training=False):
        input_ids, segment_ids = inputs
        seq_len = tf.shape(input_ids)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        x = (self.token_embedding(input_ids)
             + self.position_embedding(positions)[tf.newaxis, :, :]
             + self.segment_embedding(segment_ids))
        x = self.norm(x)
        return self.dropout(x, training=training)


class BertEncoderBlock(layers.Layer):
    def __init__(self, hidden_size, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=hidden_size // num_heads,
            dropout=dropout
        )
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation=tf.nn.gelu),
            layers.Dense(hidden_size)
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-12)
        self.norm2 = layers.LayerNormalization(epsilon=1e-12)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, attention_mask=None, training=False):
        # Bidirectional attention: each token can attend to tokens on either side.
        attn_mask = None
        if attention_mask is not None:
            # Shape: (batch, query_length, key_length)
            attn_mask = tf.cast(attention_mask[:, tf.newaxis, :], tf.bool)

        attn = self.attention(
            query=x, value=x, key=x,
            attention_mask=attn_mask,
            training=training
        )
        x = self.norm1(x + self.drop1(attn, training=training))
        ffn = self.ffn(x, training=training)
        return self.norm2(x + self.drop2(ffn, training=training))


def build_bert(vocab_size, max_len, hidden_size=128, num_heads=4,
               ff_dim=256, num_layers=2):
    token_inputs = layers.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
    segment_inputs = layers.Input(shape=(max_len,), dtype=tf.int32, name="segment_ids")
    mask_inputs = layers.Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")

    x = BertEmbeddings(vocab_size, max_len, hidden_size)(
        [token_inputs, segment_inputs]
    )
    for _ in range(num_layers):
        x = BertEncoderBlock(hidden_size, num_heads, ff_dim)(
            x, attention_mask=mask_inputs
        )

    # MLM prediction for every token position
    mlm_output = layers.Dense(vocab_size, name="mlm_head")(x)

    # NSP prediction from the [CLS] token representation
    cls_output = layers.Lambda(lambda t: t[:, 0, :], name="cls_representation")(x)
    cls_output = layers.Dense(hidden_size, activation="tanh")(cls_output)
    nsp_output = layers.Dense(1, activation="sigmoid", name="nsp_head")(cls_output)

    model = tf.keras.Model(
        inputs=[token_inputs, segment_inputs, mask_inputs],
        outputs={"mlm_head": mlm_output, "nsp_head": nsp_output}
    )
    return model

HIDDEN_SIZE = 128
NUM_HEADS = 4
FF_DIM = 256
NUM_LAYERS = 2

bert = build_bert(VOCAB_SIZE, MAX_LEN, HIDDEN_SIZE, NUM_HEADS, FF_DIM, NUM_LAYERS)
bert.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ segment_ids         │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_embeddings     │ (None, 64, 128)   │    785,920 │ input_ids[0][0],  │
│ (BertEmbeddings)    │                   │            │ segment_ids[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_mask      │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_encoder_block  │ (None, 64, 128)   │    132,480 │ bert_embeddings[… │
│ (BertEncoderBlock)  │                   │            │ attention_mask[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_encoder_block… │ (None, 64, 128)   │    132,480 │ bert_encoder_blo… │
│ (BertEncoderBlock)  │                   │            │ attention_mask[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_representation  │ (None, 128)       │          0 │ bert_encoder_blo… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │     16,512 │ cls_representati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlm_head (Dense)    │ (None, 64, 6072)  │    783,288 │ bert_encoder_blo… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nsp_head (Dense)    │ (None, 1)         │        129 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,850,809 (7.06 MB)

 Trainable params: 1,850,809 (7.06 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Train using both objectives

The MLM loss ignores positions whose label is `-100`. NSP uses binary cross-entropy. We use a custom training step so ignored MLM positions do not contribute to the loss.

In [6]:
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4)
nsp_loss_fn = tf.keras.losses.BinaryCrossentropy()
train_loss_metric = tf.keras.metrics.Mean(name="train_loss")
val_loss_metric = tf.keras.metrics.Mean(name="val_loss")

@tf.function
def train_step(input_batch, segment_batch, mask_batch, mlm_batch, nsp_batch):
    with tf.GradientTape() as tape:
        outputs = bert(
            [input_batch, segment_batch, mask_batch],
            training=True
        )
        mlm_logits = outputs["mlm_head"]
        nsp_probs = outputs["nsp_head"]

        # Replace -100 labels before sparse CE, then use a mask to ignore them.
        valid_positions = tf.cast(tf.not_equal(mlm_batch, -100), tf.float32)
        safe_labels = tf.where(mlm_batch == -100, 0, mlm_batch)
        per_token_loss = tf.keras.losses.sparse_categorical_crossentropy(
            safe_labels, mlm_logits, from_logits=True
        )
        mlm_loss = tf.reduce_sum(per_token_loss * valid_positions) / (
            tf.reduce_sum(valid_positions) + 1e-8
        )
        nsp_loss = nsp_loss_fn(nsp_batch, nsp_probs)
        total_loss = mlm_loss + nsp_loss

    grads = tape.gradient(total_loss, bert.trainable_variables)
    optimizer.apply_gradients(zip(grads, bert.trainable_variables))
    return total_loss, mlm_loss, nsp_loss


def run_epoch(data, training=True, batch_size=16):
    ids, segs, masks, mlm, nsp = data
    order = np.arange(len(ids))
    if training:
        np.random.shuffle(order)
    losses = []

    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        batch = (ids[idx], segs[idx], masks[idx], mlm[idx], nsp[idx])
        if training:
            total, mlm_l, nsp_l = train_step(*[tf.convert_to_tensor(x) for x in batch])
            losses.append([float(total), float(mlm_l), float(nsp_l)])
        else:
            outputs = bert([batch[0], batch[1], batch[2]], training=False)
            valid = (batch[3] != -100).astype(np.float32)
            safe = np.where(batch[3] == -100, 0, batch[3])
            logits = outputs["mlm_head"].numpy()
            # Compute masked-token loss only
            per_token = tf.keras.losses.sparse_categorical_crossentropy(
                safe, logits, from_logits=True
            ).numpy()
            mlm_l = float((per_token * valid).sum() / (valid.sum() + 1e-8))
            nsp_l = float(nsp_loss_fn(batch[4], outputs["nsp_head"]).numpy())
            losses.append([mlm_l + nsp_l, mlm_l, nsp_l])

    return np.mean(losses, axis=0)

EPOCHS = 3
BATCH_SIZE = 16

for epoch in range(EPOCHS):
    train_losses = run_epoch(train_data, training=True, batch_size=BATCH_SIZE)
    if len(val_data[0]) > 0:
        val_losses = run_epoch(val_data, training=False, batch_size=BATCH_SIZE)
        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"train total={train_losses[0]:.4f} (MLM={train_losses[1]:.4f}, NSP={train_losses[2]:.4f}) | "
            f"val total={val_losses[0]:.4f} (MLM={val_losses[1]:.4f}, NSP={val_losses[2]:.4f})"
        )
    else:
        print(f"Epoch {epoch + 1}/{EPOCHS} | train losses: {train_losses}")

Epoch 1/3 | train total=7.2067 (MLM=6.4967, NSP=0.7100) | val total=6.7827 (MLM=6.0888, NSP=0.6939)
Epoch 2/3 | train total=6.6239 (MLM=5.9261, NSP=0.6978) | val total=6.8005 (MLM=6.1057, NSP=0.6947)
Epoch 3/3 | train total=6.5204 (MLM=5.8291, NSP=0.6913) | val total=6.7595 (MLM=6.0637, NSP=0.6958)


## 7. Inspect one masked-token prediction

This is a small demonstration of the MLM head. It does not generate a paragraph like a GPT model; BERT is an encoder model trained to represent context and predict masked tokens.

In [ ]:
def show_masked_prediction(sentence):
    words = tokenize(sentence)
    ids = [CLS_ID] + [token_to_id.get(w, UNK_ID) for w in words] + [SEP_ID]
    if len(ids) > MAX_LEN:
        ids = ids[:MAX_LEN]
        ids[-1] = SEP_ID

    # Mask one ordinary token, not [CLS] or [SEP]
    candidate_positions = [i for i in range(1, len(ids) - 1)
                           if ids[i] not in SPECIAL_IDS]
    if not candidate_positions:
        print("No suitable token to mask.")
        return

    pos = candidate_positions[len(candidate_positions) // 2]
    original = ids[pos]
    masked_ids = ids.copy()
    masked_ids[pos] = MASK_ID

    pad = MAX_LEN - len(masked_ids)
    masked_ids += [PAD_ID] * pad
    seg = [0] * MAX_LEN
    attn = [1] * (MAX_LEN - pad) + [0] * pad

    out = bert(
        [np.array([masked_ids]), np.array([seg]), np.array([attn])],
        training=False
    )
    logits = out["mlm_head"].numpy()[0, pos]
    top_ids = np.argsort(logits)[-5:][::-1]

    display_tokens = [id_to_token.get(i, UNK) for i in ids[:len(ids)-pad]]
    display_tokens[pos] = "[MASK]"
    print("Masked input:", " ".join(display_tokens))
    print("Original token:", id_to_token.get(original, UNK))
    print("Top predictions:")
    for idx in top_ids:
        print(f"  {id_to_token.get(int(idx), UNK):<15} score={logits[idx]:.3f}")

show_masked_prediction("Harry looked at the door.")

## 8. Predict whether a sentence pair is consecutive (NSP)

The NSP output is a probability between 0 and 1. This model's predictions are only a learning demonstration because the corpus and training are small.

In [7]:
def predict_nsp(sentence_a, sentence_b):
    a_ids = encode_words(sentence_a)
    b_ids = encode_words(sentence_b)

    while len(a_ids) + len(b_ids) > MAX_LEN - 3:
        if len(a_ids) > len(b_ids):
            a_ids.pop()
        else:
            b_ids.pop()

    ids = [CLS_ID] + a_ids + [SEP_ID] + b_ids + [SEP_ID]
    segs = [0] * (len(a_ids) + 2) + [1] * (len(b_ids) + 1)
    mask = [1] * len(ids)
    pad = MAX_LEN - len(ids)

    ids += [PAD_ID] * pad
    segs += [0] * pad
    mask += [0] * pad

    probability = float(bert(
        [np.array([ids]), np.array([segs]), np.array([mask])],
        training=False
    )["nsp_head"].numpy()[0, 0])

    print(f"Sentence A: {sentence_a}")
    print(f"Sentence B: {sentence_b}")
    print(f"P(IsNext) = {probability:.3f}")
    print("Prediction:", "IsNext" if probability >= 0.5 else "NotNext")

predict_nsp("Harry looked at the door.", "He slowly opened it.")

Sentence A: Harry looked at the door.
Sentence B: He slowly opened it.
P(IsNext) = 0.748
Prediction: IsNext


## 9. Save the model

This saves the trained Keras model. The tokenizer vocabulary should also be saved because the model depends on the same token-to-ID mapping.

In [8]:
bert.save("bert_tiny_mlm_nsp.keras")

with open("bert_vocab.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(vocab))

print("Saved: bert_tiny_mlm_nsp.keras and bert_vocab.txt")

Saved: bert_tiny_mlm_nsp.keras and bert_vocab.txt
